# PFE ML — Library Shootout (Phase A)

**Question this notebook answers:** *Did we land on the best model family with one HGB run, or were we lucky?*

Trains four gradient-boosting libraries on the **same** 2M-row period-fixed dataset, the **same** 2017–2023 train / 2024 test split, and **comparable default hyperparameters** (~400 boosting rounds, learning rate 0.05, mild L2 regularization, class-imbalance handled via each library's native knob). The only thing that changes between runs is the library.

## What this is, methodologically

This is **Phase A** of a three-phase model-selection study:

| Phase | Question | This notebook |
|---|---|---|
| **A. Library comparison** | Which boosting library wins at default settings? | **Yes — this one.** |
| B. Hyperparameter tuning | Can we improve the top-2 libraries with a random search? | Follow-up notebook |
| C. Temporal stability | Does the winner hold up with 2023 as test (instead of 2024)? | Follow-up notebook |
| D. Interpretability | What does the winner actually use? (SHAP analysis) | Thesis chapter |

Keeping the four phases separate is what makes the contributions individually attributable.

## Honest caveats — read these before interpreting the results

- **Single time-split, no CV.** A library that wins here by 0.5 pp AUC might lose under TimeSeriesSplit. Treat differences below ~1 pp AUC as noise.
- **Default hyperparameters.** Each library is matched on rounds / LR / depth, not tuned. The winner of Phase A is *not necessarily* the winner after Phase B.
- **Class-imbalance handling differs.** HGB and LightGBM use `class_weight='balanced'`; XGBoost uses `scale_pos_weight = N_neg / N_pos`; CatBoost uses `class_weights=[1, N_neg/N_pos]`. Mathematically equivalent for binary classification, but not literally the same line of code.
- **Reproducibility.** Each library uses `random_state=42`. Boosting on the same data with the same seed is deterministic across runs.

## What's needed on the branch

Before running, commit + push these to `data-extraction`:
1. `app/tools/train_continuity_model.py` — now exposes `--model-family {hgb,lightgbm,catboost,xgboost}` and a `_build_model_pipeline()` dispatcher.
2. `collabs/requirements-colab.txt` — adds `lightgbm`, `xgboost`, `catboost` (the install adds ~2 min to section 2).

## 1. Runtime And Constants

**High-RAM CPU.** Each library trains in 5–10 min; permutation importance adds 2–3 min per run. Total ~40–50 min for four libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'

MODEL_FAMILIES = ['hgb', 'lightgbm', 'catboost', 'xgboost']

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('BRANCH       =', BRANCH)
print('TRAIN_CAP    =', TRAIN_MAX_ROWS)
print('LIBRARIES    =', MODEL_FAMILIES)

## 2. Pull Code And Install Dependencies

Make sure the `--model-family` plumbing and the three new library pins are committed and pushed to `data-extraction` first.

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import importlib, sys

train_script_text = (Path(BACKEND_DIR) / 'app' / 'tools' / 'train_continuity_model.py').read_text(encoding='utf-8')
code_checks = {
    '--model-family CLI flag': '--model-family' in train_script_text,
    '_build_model_pipeline dispatcher': '_build_model_pipeline' in train_script_text,
    'MODEL_FAMILIES tuple': "MODEL_FAMILIES = (" in train_script_text,
    'CategoricalCaster present': 'class CategoricalCaster' in train_script_text,
    'StringCaster present': 'class StringCaster' in train_script_text,
}
print('Source-code readiness:')
for label, ok in code_checks.items():
    print(f'  {"OK " if ok else "FAIL"}  {label}')
if not all(code_checks.values()):
    raise SystemExit('Multi-library training plumbing is missing on the pulled branch. Commit + push and re-run.')

library_checks = {}
for module in ('lightgbm', 'catboost', 'xgboost'):
    try:
        mod = importlib.import_module(module)
        library_checks[module] = getattr(mod, '__version__', '?')
    except Exception as exc:
        library_checks[module] = f'IMPORT FAILED: {exc}'
print('\nLibrary install:')
for name, version in library_checks.items():
    print(f'  {name:10s}  {version}')
if any('IMPORT FAILED' in str(v) for v in library_checks.values()):
    raise SystemExit('One of the boosting libraries failed to import. Check the pip install output above.')

## 3. Verify Data Is In Place

In [ ]:
import json, duckdb

manifest_path = Path(f'{DATA_LAKE}/clean/company_identity/_manifest.json')
if not manifest_path.exists():
    raise SystemExit(f'Missing manifest at {manifest_path}. Run pfe_ml_colab_period_fix_rebuild.ipynb first.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest.get('schema_version') == 2 and 'period' in manifest.get('grain', ''), 'Clean layer is not period-fixed.'

features_root = Path(f'{DATA_LAKE}/features/company_year_features')
labels_root = Path(f'{DATA_LAKE}/features/risk_labels')
if not list(features_root.rglob('*.parquet')) or not list(labels_root.rglob('*.parquet')):
    raise SystemExit('Feature or label parquet missing on Drive. Re-run the period-fix rebuild notebook.')

con = duckdb.connect()
counts = con.execute(f'''
    SELECT
        (SELECT count(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS feature_rows,
        (SELECT count(*) FROM read_parquet('{LABELS_GLOB}',   union_by_name=true)) AS label_rows
''').df()
print('Feature/label counts:')
print(counts.to_string(index=False))
con.close()

## 4. Train All Four Libraries

Each library uses the same 2M-row dataset, same time split, same comparable defaults. The training script appends each run to `model_run_comparison.csv` automatically, so by the end of this cell the CSV has 4 new rows (one per library).

**Expected total runtime: 30–50 min.** Watch for surprises in the per-library times printed below — if CatBoost or XGBoost takes 3× longer than HGB, that's worth noting in the thesis (training cost is a real deployment consideration).

In [ ]:
import shlex, subprocess, sys, time

run_results = {}
for family in MODEL_FAMILIES:
    print('=' * 70)
    print(f'Training {family}')
    print('=' * 70)
    train_cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
        '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
        '--target', TARGET,
        '--train-start-year', str(START_YEAR),
        '--train-end-year', str(END_YEAR),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
        '--model-family', family,
    ]
    print(' '.join(shlex.quote(p) for p in train_cmd))
    start = time.time()
    subprocess.run(train_cmd, check=True)
    elapsed = time.time() - start
    metadata_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json'
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    run_results[family] = {
        'run_name': metadata.get('run_name'),
        'run_dir': metadata.get('run_artifacts_dir'),
        'elapsed_seconds': elapsed,
        'metrics': metadata.get('metrics', {}),
    }
    metrics = metadata.get('metrics', {})
    print(
        f'\nDone ({elapsed:.0f}s). '
        f"AUC={metrics.get('roc_auc'):.4f}  "
        f"AP={metrics.get('average_precision'):.4f}  "
        f"F1@0.5={metrics.get('f1_at_0_5'):.4f}\n"
    )

print('All four libraries trained.')

## 5. Head-To-Head Comparison Table

In [ ]:
import pandas as pd

rows = []
for family in MODEL_FAMILIES:
    info = run_results[family]
    m = info['metrics']
    top_k = {item['segment']: item for item in m.get('top_k_analysis', [])}
    rows.append({
        'library': family,
        'training_seconds': round(info['elapsed_seconds']),
        'roc_auc': m.get('roc_auc'),
        'average_precision': m.get('average_precision'),
        'precision_at_0_5': m.get('precision_at_0_5'),
        'recall_at_0_5': m.get('recall_at_0_5'),
        'f1_at_0_5': m.get('f1_at_0_5'),
        'top_0.1pct_precision': top_k.get('top_0.1%', {}).get('precision'),
        'top_0.1pct_lift': top_k.get('top_0.1%', {}).get('lift'),
        'top_1pct_lift': top_k.get('top_1.0%', {}).get('lift'),
        'top_10pct_lift': top_k.get('top_10.0%', {}).get('lift'),
    })
shootout = pd.DataFrame(rows).sort_values('average_precision', ascending=False).reset_index(drop=True)
print('Library shootout (sorted by average precision):')
print(shootout.to_string(index=False))

shootout_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'library_shootout_phase_a.csv'
shootout.to_csv(shootout_csv, index=False)
print(f'\nSaved comparison CSV to: {shootout_csv}')

## 6. Side-By-Side Plots

ROC curves, PR curves, and precision-at-top-K for the four libraries on one figure each. This is what goes into the thesis chapter on model selection.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve

shootout_dir = Path(DRIVE_ROOT) / 'ml-artifacts' / 'library_shootout_phase_a'
shootout_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for family in MODEL_FAMILIES:
    metrics = run_results[family]['metrics']
    thresholds = metrics.get('threshold_analysis', [])
    top_k = metrics.get('top_k_analysis', [])
    label_auc = f"{family} (AUC={metrics.get('roc_auc'):.3f})"
    label_ap = f"{family} (AP={metrics.get('average_precision'):.3f})"

    pr_thr = [float(r['precision']) for r in thresholds]
    re_thr = [float(r['recall']) for r in thresholds]
    axes[0].plot(re_thr, pr_thr, marker='o', label=label_ap)

    fpr_seq = [1 - float(r.get('false_positive_rate', 0)) for r in thresholds]
    tpr_seq = re_thr
    fpr_actual = [float(r.get('false_positive_rate', 0)) for r in thresholds]
    axes[1].plot(fpr_actual, tpr_seq, marker='o', label=label_auc)

    fractions = [float(item['fraction']) for item in top_k]
    precisions = [float(item['precision']) for item in top_k]
    axes[2].plot(fractions, precisions, marker='o', label=family)

axes[0].set_title('Precision vs Recall (from threshold sweep)')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot([0, 1], [0, 1], '--', color='gray', linewidth=0.8)
axes[1].set_title('ROC (from threshold sweep)')
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].grid(True, alpha=0.3); axes[1].legend()

axes[2].set_xscale('log')
axes[2].set_title('Precision at top-K (operational view)')
axes[2].set_xlabel('Top fraction flagged'); axes[2].set_ylabel('Precision')
axes[2].grid(True, alpha=0.3); axes[2].legend()

fig.suptitle(f'Library shootout — 2M rows, time-split 2024, comparable defaults')
fig.tight_layout(rect=[0, 0, 1, 0.96])
comparison_png = shootout_dir / 'pr_roc_topk_comparison.png'
fig.savefig(comparison_png, dpi=160)
plt.show()
print(f'Saved figure to: {comparison_png}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_positions = list(range(len(MODEL_FAMILIES)))
aucs = [shootout[shootout['library'] == f]['roc_auc'].iloc[0] for f in MODEL_FAMILIES]
aps = [shootout[shootout['library'] == f]['average_precision'].iloc[0] for f in MODEL_FAMILIES]
times = [run_results[f]['elapsed_seconds'] for f in MODEL_FAMILIES]

axes[0].bar(x_positions, aucs, color=['#0f766e', '#1d4ed8', '#b45309', '#7c3aed'])
axes[0].set_xticks(x_positions, MODEL_FAMILIES)
axes[0].set_ylabel('ROC AUC'); axes[0].set_title('ROC AUC by library')
axes[0].set_ylim(0.7, max(aucs) * 1.02)
for x, value in zip(x_positions, aucs):
    axes[0].text(x, value + 0.002, f'{value:.4f}', ha='center', fontsize=10)

axes[1].bar(x_positions, times, color=['#0f766e', '#1d4ed8', '#b45309', '#7c3aed'])
axes[1].set_xticks(x_positions, MODEL_FAMILIES)
axes[1].set_ylabel('Training seconds'); axes[1].set_title('Training time by library')
for x, value in zip(x_positions, times):
    axes[1].text(x, value + max(times) * 0.01, f'{value:.0f}s', ha='center', fontsize=10)

fig.tight_layout()
bar_png = shootout_dir / 'auc_and_time_by_library.png'
fig.savefig(bar_png, dpi=160)
plt.show()
print(f'Saved figure to: {bar_png}')

## 7. Display Each Run's Own Summary

Drops a markdown-rendered `run_summary.md` for each of the four runs so you can scan the per-library breakdowns side by side.

In [ ]:
from IPython.display import Markdown, display

for family in MODEL_FAMILIES:
    run_dir = Path(run_results[family]['run_dir'])
    summary_path = run_dir / 'run_summary.md'
    print(f'\n## {family} — {run_dir.name}')
    if summary_path.exists():
        display(Markdown(summary_path.read_text(encoding='utf-8')))
    else:
        print(f'  (run_summary.md missing at {summary_path})')

## 8. Append To The Master Run Index

`model_run_comparison.csv` was appended four times during section 4 (once per library), so the file is automatically up to date. This cell just prints the latest rows for confirmation, plus the long-running chronological view back to Run 1.

In [ ]:
comparison_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.csv'
if comparison_csv.exists():
    runs_summary = pd.read_csv(comparison_csv)
    keep = [
        'run_name', 'model_family', 'rows', 'feature_count',
        'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
    ]
    keep = [c for c in keep if c in runs_summary.columns]
    print('All runs to date (chronological):')
    print(runs_summary[keep].to_string(index=False))
    print('\nLast four rows are the library shootout.')
else:
    print('No model_run_comparison.csv yet.')

## 9. What To Send / What To Decide

### Attach to the thesis
- `ml-artifacts/library_shootout_phase_a.csv` — the headline comparison table.
- `ml-artifacts/library_shootout_phase_a/pr_roc_topk_comparison.png` — side-by-side PR / ROC / top-K curves.
- `ml-artifacts/library_shootout_phase_a/auc_and_time_by_library.png` — AUC vs training-time tradeoff.
- The four newest `runs/*` folders — each library's own `run_summary.md`, `feature_importances.csv`, and threshold plots.
- The updated `model_run_comparison.csv` (now 9 + 4 = 13 rows).

### Decide before Phase B
- **Pick the top 2 libraries** by average precision (ties broken by F1 @ 0.5). Those are the candidates for hyperparameter tuning.
- **Note the training-time tradeoff.** If two libraries tie on AP but one trains in half the time, the faster one is the practical pick — especially if you'll be retraining periodically in production.
- **If one library is more than 2 pp AP behind the leader**, drop it from Phase B. Don't waste tuning budget on it.

### Frame in the report
*"At default hyperparameters and with class-imbalance handling enabled in each library's native form, [WINNER] achieved the highest average precision (X.XX) on the held-out 2024 test set, exceeding the runner-up [SECOND] by Y.Y pp. The result is consistent with [WINNER]'s reputation for handling [high-cardinality categoricals / heavy missingness / etc.]. Hyperparameter tuning in Phase B is restricted to [WINNER] and [SECOND]."*